# German Credit Data - EDA

Exploring the dataset before building the training pipeline. 1000 applicants, 20 features, binary classification (good/bad credit risk).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [2]:
data = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
df = data.frame

print(f'Shape: {df.shape}')
print(f'
Columns: {", ".join(df.columns)}')

Shape: (1000, 21)

Columns: checking_status, duration, credit_history, purpose, credit_amount, savings_status, employment, installment_rate, personal_status, other_parties, residence_since, property_magnitude, age, other_payment_plans, housing, existing_credits, job, num_dependents, telephone, foreign_worker, class


In [3]:
print('Target distribution:')
for val, count in df['class'].value_counts().items():
    print(f'{val:7s} {count} ({count/len(df)*100:.1f}%)')

Target distribution:
good    700 (70.0%)
bad     300 (30.0%)


In [4]:
fig, ax = plt.subplots()
df['class'].value_counts().plot.bar(color=['#4CAF50', '#F44336'], ax=ax)
ax.set_title('Credit Risk Distribution')
ax.set_ylabel('Count')
ax.set_xticklabels(['Good (0)', 'Bad (1)'], rotation=0)
plt.tight_layout()
plt.show()

<Figure size 1000x600 with 1 Axes>

70/30 split, moderately imbalanced. Not as extreme as fraud detection but still worth accounting for.

In [5]:
numeric_cols = ['duration', 'credit_amount', 'age', 'installment_rate',
                'existing_credits', 'num_dependents']

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, col in enumerate(numeric_cols):
    ax = axes[i // 3, i % 3]
    for label, color in [('good', '#4CAF50'), ('bad', '#F44336')]:
        subset = df[df['class'] == label][col].astype(float)
        ax.hist(subset, bins=25, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle('Numeric Feature Distributions by Risk', y=1.02)
plt.tight_layout()
plt.show()

<Figure size 1200x800 with 6 Axes>

Bad credit applicants tend to have longer durations and higher credit amounts. Age distribution is similar for both classes, interesting.

In [6]:
# Correlation heatmap for numeric features
numeric_df = df[numeric_cols + ['class']].copy()
numeric_df['class'] = (numeric_df['class'] == 'bad').astype(int)
for col in numeric_cols:
    numeric_df[col] = pd.to_numeric(numeric_df[col], errors='coerce')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

<Figure size 1000x800 with 2 Axes>

In [7]:
corr_with_target = numeric_df.corr()['class'].drop('class').sort_values(ascending=False)
print('Correlation with target (bad credit risk):
')
for feat, val in corr_with_target.items():
    print(f'{feat:22s} {val:6.3f}')

Correlation with target (bad credit risk):

duration             0.219
credit_amount        0.162
installment_rate     0.075
num_dependents      -0.024
existing_credits    -0.010
age                 -0.041


Duration and credit amount have the strongest linear correlation with bad risk, but correlations are weak overall. The categorical features (checking_status, credit_history) probably carry more signal.

In [8]:
# Key categorical feature: checking_status
ct = pd.crosstab(df['checking_status'], df['class'], normalize='columns') * 100

print('checking_status vs risk:
')
for status in ct.index:
    print(f'{status:15s} {ct.loc[status, "good"]:5.1f}% good, {ct.loc[status, "bad"]:5.1f}% bad')

checking_status vs risk:

no checking    39.4% good, 14.7% bad
<0             27.4% good, 39.3% bad
0<=X<200       25.3% good, 32.3% bad
>=200           7.9% good, 13.7% bad


Having no checking account is actually a strong indicator of good credit in this dataset (German context, 1990s). The negative balance (<0) checking status is much more common among bad risks.

Missing values check:

In [9]:
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print('No missing values in any column.')
else:
    print(null_counts[null_counts > 0])

No missing values in any column.


Clean dataset, no nulls. Plan:
1. Feature engineering: derived features (amount_per_month, age bins, credit_load)
2. StandardScaler for numeric, OneHotEncoder for categorical
3. Compare LogReg, RF, XGBoost, LightGBM with MLflow tracking
4. SHAP for interpretability

# some features are redundant, will check VIF later